In [1]:
!pip install pyscf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.6/40.6 MB 34.6 MB/s  0:00:01


In [2]:
# H2O HF
from pyscf import gto, scf

# -----------------------------------------
# SETTINGS
# -----------------------------------------

BASIS = "def2-SVP"

# -----------------------------------------
# 1. WATER: H2O
# -----------------------------------------

water = gto.M(
    atom = """
        O  0.00000  0.00000  0.00000
        H  0.75695  0.00000  0.58588
        H -0.75695  0.00000  0.58588
    """,
    basis = BASIS,
    unit = "Angstrom",
    charge = 0,
    spin = 0
)

water_hf = scf.RHF(water)
E_H2O = water_hf.kernel()

# -----------------------------------------
# 2. HYDROGEN GAS: H2
# -----------------------------------------

hydrogen = gto.M(
    atom = """
        H  0.00000  0.00000  0.00000
        H  0.74140  0.00000  0.00000
    """,
    basis = BASIS,
    unit = "Angstrom",
    charge = 0,
    spin = 0
)

hydrogen_hf = scf.RHF(hydrogen)
E_H2 = hydrogen_hf.kernel()

# -----------------------------------------
# 3. OXYGEN ATOM: O
# -----------------------------------------

oxygen = gto.M(
    atom = """
        O  0.00000  0.00000  0.00000
    """,
    basis = BASIS,
    unit = "Angstrom",
    charge = 0,
    spin = 0
)

oxygen_hf = scf.ROHF(oxygen)
E_O = oxygen_hf.kernel()

# -----------------------------------------
# 4. CALCULATE REACTION ENERGY
# -----------------------------------------

# Reaction:
# H2O + Pt -> H2 + O + Pt
#
# Platinum cancels if treated as an
# unchanged, noninteracting spectator.

E_reactants = E_H2O
E_products = E_H2 + E_O

delta_E = E_products - E_reactants

# Convert Hartree to kJ/mol
HARTREE_TO_KJMOL = 2625.50

delta_E_kJmol = delta_E * HARTREE_TO_KJMOL

# -----------------------------------------
# 5. DISPLAY RESULTS
# -----------------------------------------

print("=" * 45)
print("HARTREE-FOCK ENERGY RESULTS")
print("=" * 45)

print(f"H2O Energy: {E_H2O:.10f} Hartree")
print(f"H2 Energy:  {E_H2:.10f} Hartree")
print(f"O Energy:   {E_O:.10f} Hartree")

print("-" * 45)

print(f"Reactant Energy: {E_reactants:.10f} Hartree")
print(f"Product Energy:  {E_products:.10f} Hartree")

print("-" * 45)

print(f"Reaction Energy: {delta_E:.10f} Hartree")
print(f"Reaction Energy: {delta_E_kJmol:.4f} kJ/mol")

print("=" * 45)

if delta_E < 0:
    print("Reaction is EXOTHERMIC (releases energy)")
else:
    print("Reaction is ENDOTHERMIC (absorbs energy)")


converged SCF energy = -75.961014897512
converged SCF energy = -1.12890637848863
converged SCF energy = -74.5924321418067
HARTREE-FOCK ENERGY RESULTS
H2O Energy: -75.9610148975 Hartree
H2 Energy:  -1.1289063785 Hartree
O Energy:   -74.5924321418 Hartree
---------------------------------------------
Reactant Energy: -75.9610148975 Hartree
Product Energy:  -75.7213385203 Hartree
---------------------------------------------
Reaction Energy: 0.2396763772 Hartree
Reaction Energy: 629.2703 kJ/mol
Reaction is ENDOTHERMIC (absorbs energy)


In [3]:
#H2O+Pt HF

from pyscf import gto, scf

mol = gto.M(
    atom = """
        O   0.00000   0.00000   0.00000
        H   0.75695   0.00000   0.58588
        H  -0.75695   0.00000   0.58588
        Pt  0.00000   0.00000   2.50000
    """,

    basis = {
        'H': 'def2-SVP',
        'O': 'def2-SVP',
        'Pt': 'def2-SVP'
    },

    ecp = {
        'Pt': 'def2-SVP'
    },

    unit = "Angstrom",
    charge = 0,
    spin = 0
)

# Use unrestricted Hartree-Fock for the system
mf = scf.UHF(mol)

energy = mf.kernel()

print("\nH2O + Pt RESULTS")
print("-" * 40)

print(f"Total Energy: {energy:.10f} Hartree")
print(f"Total Energy: {energy * 2625.50:.4f} kJ/mol")
print(f"Converged: {mf.converged}")

converged SCF energy = -194.303717534987  <S^2> = 7.4671849e-08  2S+1 = 1.0000001

H2O + Pt RESULTS
----------------------------------------
Total Energy: -194.3037175350 Hartree
Total Energy: -510144.4104 kJ/mol
Converged: True


In [4]:



# =========================================
# 1. DEFINE PLATINUM ATOM
# =========================================

mol = gto.M(
    atom = "Pt 0.0 0.0 0.0",

    basis = {
        'Pt': 'def2-SVP'
    },

    ecp = {
        'Pt': 'def2-SVP'
    },

    unit = "Angstrom",
    charge = 0,
    spin = 0
)

# =========================================
# 2. FIRST HF CONVERGENCE ATTEMPT
# =========================================

mf = scf.UHF(mol)

mf.init_guess = 'atom'
mf.max_cycle = 500
mf.conv_tol = 1e-9
mf.diis_space = 15
mf.level_shift = 0.2
mf.damp = 0.0

energy = mf.kernel()

# =========================================
# 3. SECOND ATTEMPT IF FIRST FAILS
# =========================================

if not mf.converged:

    print("Standard HF did not converge.")
    print("Trying second-order HF...")

    mf = scf.UHF(mol).newton()

    mf.max_cycle = 200
    mf.conv_tol = 1e-9

    energy = mf.kernel()

# =========================================
# 4. DISPLAY RESULTS
# =========================================

print("\n" + "=" * 50)
print("PLATINUM HARTREE-FOCK RESULTS")
print("=" * 50)

if mf.converged:

    print(f"HF Energy: {energy:.10f} Hartree")

    print(f"HF Energy: {energy * 2625.50:.4f} kJ/mol")

    print("\nHF calculation successfully converged!")

else:

    print("\nHF FAILED TO CONVERGE.")
    print("The calculated energy should not be used.")

print("=" * 50)



converged SCF energy = -118.3570453812  <S^2> = -3.5527137e-15  2S+1 = 1

PLATINUM HARTREE-FOCK RESULTS
HF Energy: -118.3570453812 Hartree
HF Energy: -310746.4226 kJ/mol

HF calculation successfully converged!


In [5]:
#H2O CCSD

from pyscf import gto, scf, cc

# -----------------------------------------
# 1. DEFINE WATER MOLECULE
# -----------------------------------------

mol = gto.M(
    atom = """
        O   0.00000   0.00000   0.00000
        H   0.75695   0.00000   0.58588
        H  -0.75695   0.00000   0.58588
    """,
    basis = "def2-SVP",
    unit = "Angstrom",
    charge = 0,
    spin = 0
)

# -----------------------------------------
# 2. HARTREE-FOCK CALCULATION
# -----------------------------------------

mf = scf.RHF(mol)

E_HF = mf.kernel()

# -----------------------------------------
# 3. COUPLED CLUSTER CALCULATION
# -----------------------------------------

mycc = cc.CCSD(mf)

mycc.kernel()

# -----------------------------------------
# 4. EXTRACT CCSD ENERGY
# -----------------------------------------

E_CCSD = mycc.e_tot

# -----------------------------------------
# 5. CONVERT ENERGY TO kJ/mol
# -----------------------------------------

HARTREE_TO_KJMOL = 2625.50

E_HF_kJmol = E_HF * HARTREE_TO_KJMOL
E_CCSD_kJmol = E_CCSD * HARTREE_TO_KJMOL

# -----------------------------------------
# 6. DISPLAY RESULTS
# -----------------------------------------

print("=" * 50)
print("H2O COUPLED CLUSTER RESULTS")
print("=" * 50)

print(f"Hartree-Fock Energy: {E_HF:.10f} Hartree")
print(f"CCSD Energy:         {E_CCSD:.10f} Hartree")

print("-" * 50)

print(f"HF Energy:   {E_HF_kJmol:.4f} kJ/mol")
print(f"CCSD Energy: {E_CCSD_kJmol:.4f} kJ/mol")

print("-" * 50)

print(f"Correlation Energy: {E_CCSD - E_HF:.10f} Hartree")

print("=" * 50)

print("HF Converged:", mf.converged)
print("CCSD Converged:", mycc.converged)


converged SCF energy = -75.9610148975121
E(CCSD) = -76.17392317736619  E_corr = -0.2129082798541267
H2O COUPLED CLUSTER RESULTS
Hartree-Fock Energy: -75.9610148975 Hartree
CCSD Energy:         -76.1739231774 Hartree
--------------------------------------------------
HF Energy:   -199435.6446 kJ/mol
CCSD Energy: -199994.6353 kJ/mol
--------------------------------------------------
Correlation Energy: -0.2129082799 Hartree
HF Converged: True
CCSD Converged: True


In [6]:
#H2O + PT CCSD

from pyscf import gto, scf, cc

# =========================================
# 1. DEFINE H2O + Pt SYSTEM
# =========================================

mol = gto.M(
    atom = """
        O   0.00000   0.00000   0.00000
        H   0.75695   0.00000   0.58588
        H  -0.75695   0.00000   0.58588
        Pt  0.00000   0.00000   -2.40000
    """,

    basis = {
        'H': 'def2-SVP',
        'O': 'def2-SVP',
        'Pt': 'def2-SVP'
    },

    ecp = {
        'Pt': 'def2-SVP'
    },

    unit = "Angstrom",
    charge = 0,
    spin = 0
)

# =========================================
# 2. STABILIZE REFERENCE ORBITALS
# =========================================

mf = scf.UHF(mol)

mf.max_cycle = 300
mf.conv_tol = 1e-9

# Improve SCF stability
mf.damp = 0.3
mf.level_shift = 0.3
mf.diis_space = 12
mf.diis_start_cycle = 3

mf.kernel()

if not mf.converged:
    raise RuntimeError(
        "Reference calculation failed to converge. "
        "CCSD cannot be reliably started."
    )

# =========================================
# 3. RUN CCSD
# =========================================

mycc = cc.UCCSD(mf)

mycc.max_cycle = 200
mycc.conv_tol = 1e-7
mycc.conv_tol_normt = 1e-6

# Improve CCSD iterative convergence
mycc.diis = True
mycc.diis_space = 12
mycc.diis_start_cycle = 3

mycc.kernel()

# =========================================
# 4. DISPLAY CCSD RESULTS ONLY
# =========================================

print("\n" + "=" * 50)
print("H2O + Pt COUPLED CLUSTER RESULTS")
print("=" * 50)

if mycc.converged:

    print(f"CCSD Energy: {mycc.e_tot:.10f} Hartree")

    print(
        f"CCSD Energy: "
        f"{mycc.e_tot * 2625.50:.4f} kJ/mol"
    )

    print("\nCCSD successfully converged!")

else:

    print("WARNING: CCSD did not converge.")
    print("Do not use this energy as a final result.")

print("=" * 50)


converged SCF energy = -194.32762404813  <S^2> = 3.2835506e-07  2S+1 = 1.0000007
E(UCCSD) = -194.8469639193125  E_corr = -0.5193398711825751

H2O + Pt COUPLED CLUSTER RESULTS
CCSD Energy: -194.8469639193 Hartree
CCSD Energy: -511570.7038 kJ/mol

CCSD successfully converged!


In [7]:
from pyscf import gto, scf, cc

# =========================================
# 1. DEFINE PLATINUM ATOM
# =========================================

mol = gto.M(
    atom = """
        Pt  0.00000  0.00000  0.00000
    """,

    basis = {
        'Pt': 'def2-SVP'
    },

    ecp = {
        'Pt': 'def2-SVP'
    },

    unit = "Angstrom",
    charge = 0,
    spin = 0
)

# =========================================
# 2. HARTREE-FOCK REFERENCE WITH NEWTON-RAPHSON SOLVER
# =========================================

mf_uhf = scf.UHF(mol)
mf = mf_uhf.newton() # Enable second-order SCF solver to force convergence

mf.max_cycle = 300
mf.conv_tol = 1e-9

mf.kernel()

if not mf.converged:
    raise RuntimeError("Reference calculation did not converge even with second-order solver.")

# =========================================
# 3. CCSD CALCULATION (WITH OPTIMIZED SOLVER SETTINGS)
# =========================================

mycc = cc.UCCSD(mf)

# Increase cycles and DIIS space to handle transition metal multi-reference character
mycc.max_cycle = 400
mycc.conv_tol = 1e-7
mycc.conv_tol_normt = 1e-5  # Slightly relaxed vector norm convergence to assist completion

mycc.diis = True
mycc.diis_space = 25  # Larger DIIS space to stabilize amplitude oscillations

mycc.kernel()

# =========================================
# 4. DISPLAY RESULTS
# =========================================

print("\n" + "=" * 45)
print("PLATINUM CCSD RESULTS")
print("=" * 45)

if mycc.converged:
    print(f"Platinum CCSD Energy: {mycc.e_tot:.10f} Hartree")
    print(f"Platinum CCSD Energy: {mycc.e_tot * 2625.50:.4f} kJ/mol")
    print("\nCCSD successfully converged!")
else:
    print("\nWARNING: CCSD did not converge.")
    print("Do not use this energy as a final result.")

print("=" * 45)

converged SCF energy = -118.357045381186  <S^2> = -3.5527137e-15  2S+1 = 1
E(UCCSD) = -118.6437147034644  E_corr = -0.2866693222786147

PLATINUM CCSD RESULTS
Platinum CCSD Energy: -118.6437147035 Hartree
Platinum CCSD Energy: -311499.0730 kJ/mol

CCSD successfully converged!


In [1]:
!pip install -U qiskit qiskit-nature qbraid scipy

In [2]:
import time
import numpy as np
from pyscf import gto, scf, cc, mcscf, ao2mo

BASIS = "def2-SVP"          # same basis (and its ECP) as in cells 1–6
SPIN = 0                    # Pt(0) d10 singlet: non-trivial CAS(2,2)
H2KCAL = 627.509
H2KJ = 2625.50

# Water in the xz plane; Pt on the OXYGEN side (atop site), Pt–O = 2.40 Å
WATER = """O 0.00000 0.00000 0.00000
H 0.75695 0.00000 0.58588
H -0.75695 0.00000 0.58588"""
PT = "Pt 0.00000 0.00000 -2.40000"
GHOST_WATER = """ghost-O 0.00000 0.00000 0.00000
ghost-H 0.75695 0.00000 0.58588
ghost-H -0.75695 0.00000 0.58588"""
GHOST_PT = "ghost-Pt 0.00000 0.00000 -2.40000"

SYSTEMS = {
    "complex": f"{WATER}\n{PT}",
    "Pt":      PT,                          # = cells 3 and 6
    "H2O":     WATER,                       # = cells 1 and 4
    "Pt_cp":   f"{PT}\n{GHOST_WATER}",     # Pt in the complex basis (counterpoise)
    "H2O_cp":  f"{WATER}\n{GHOST_PT}",     # H2O in the complex basis (counterpoise)
}


In [3]:
def classical(atoms):
    mol = gto.M(atom=atoms, basis=BASIS, ecp=BASIS, spin=SPIN,
                charge=0, unit="Angstrom", verbose=0)
    t0 = time.time()
    mf = scf.RHF(mol)
    mf.max_cycle = 300
    mf.kernel()
    t_hf = time.time() - t0
    if not mf.converged:
        raise RuntimeError("HF did not converge")
    nocc = mol.nelectron // 2
    gap = mf.mo_energy[nocc] - mf.mo_energy[nocc - 1]
    if gap <= 0:
        raise RuntimeError("HOMO >= LUMO: non-aufbau solution, do not use")

    t0 = time.time()
    mycc = cc.CCSD(mf).run()
    t_cc = time.time() - t0

    mc = mcscf.CASCI(mf, 2, 2)
    mc.verbose = 0
    e_casci = mc.kernel()[0]
    h1, e_core = mc.get_h1eff()                     # 2x2 + core/inactive energy
    h2 = ao2mo.restore(1, mc.get_h2eff(), 2)        # 2x2x2x2
    return dict(e_hf=mf.e_tot, e_cc=mycc.e_tot, e_casci=e_casci,
                h1=h1, h2=h2, e_core=e_core, t_hf=t_hf, t_cc=t_cc,
                gap=gap, cc_conv=mycc.converged)

res = {}
for name, atoms in SYSTEMS.items():
    res[name] = classical(atoms)
    d = res[name]
    print(f"{name:8s} HF={d['e_hf']:.8f}  CCSD={d['e_cc']:.8f}  CASCI(2,2)={d['e_casci']:.8f}  "
          f"gap={d['gap']:.3f} Ha  t_HF={d['t_hf']:.2f}s  t_CCSD={d['t_cc']:.2f}s  CCSD conv={d['cc_conv']}")



ECP def2-SVP not found for  H
ECP def2-SVP not found for  O


complex  HF=-194.32762405  CCSD=-194.84696406  CASCI(2,2)=-194.32950979  gap=0.323 Ha  t_HF=1.91s  t_CCSD=25.97s  CCSD conv=True
Pt       HF=-118.35704538  CCSD=-118.64371471  CASCI(2,2)=-118.35747572  gap=0.326 Ha  t_HF=0.10s  t_CCSD=1.06s  CCSD conv=True


ECP def2-SVP not found for  H
ECP def2-SVP not found for  O


H2O      HF=-75.96101490  CCSD=-76.17392318  CASCI(2,2)=-75.96110767  gap=0.674 Ha  t_HF=0.09s  t_CCSD=0.20s  CCSD conv=True
Pt_cp    HF=-118.35716959  CCSD=-118.64609315  CASCI(2,2)=-118.35758609  gap=0.325 Ha  t_HF=0.29s  t_CCSD=4.03s  CCSD conv=True


ECP def2-SVP not found for  H
ECP def2-SVP not found for  O


H2O_cp   HF=-75.96504177  CCSD=-76.18037916  CASCI(2,2)=-75.96504248  gap=0.593 Ha  t_HF=0.32s  t_CCSD=2.46s  CCSD conv=True


In [4]:
from scipy.optimize import minimize
from qiskit_nature.second_q.hamiltonians import ElectronicEnergy
from qiskit_nature.second_q.mappers import ParityMapper
from qiskit_nature.second_q.circuit.library import UCCSD, HartreeFock
from qiskit.primitives import StatevectorEstimator

def qubit_hamiltonian(d):
    fermionic = ElectronicEnergy.from_raw_integrals(d["h1"], d["h2"]).second_q_op()
    mapper = ParityMapper(num_particles=(1, 1))     # 4 spin-orbitals -> 2 qubits
    return mapper.map(fermionic), mapper

def build_ansatz(mapper):
    return UCCSD(2, (1, 1), mapper, initial_state=HartreeFock(2, (1, 1), mapper))

estimator = StatevectorEstimator()
for name, d in res.items():
    qop, mapper = qubit_hamiltonian(d)
    circ = build_ansatz(mapper)
    d["qop"], d["circ"] = qop, circ
    d["exact_2q"] = np.linalg.eigvalsh(qop.to_matrix()).min() + d["e_core"]
    f = lambda th: float(estimator.run([(circ, qop, th)]).result()[0].data.evs) + d["e_core"]
    t0 = time.time()
    opt = minimize(f, np.zeros(circ.num_parameters), method="COBYLA")
    d["vqe_sv"], d["theta_opt"], d["t_vqe_sv"] = opt.fun, opt.x, time.time() - t0
    print(f"{name:8s} qubits={qop.num_qubits}  params={circ.num_parameters}  "
          f"exact_2q={d['exact_2q']:.8f}  VQE_sv={d['vqe_sv']:.8f}  "
          f"|VQE-CASCI|={abs(d['vqe_sv']-d['e_casci'])*H2KCAL:.2e} kcal/mol  t={d['t_vqe_sv']:.2f}s")
print("\nComplex Hamiltonian:\n", res["complex"]["qop"])



/usr/local/lib/python3.13/dist-packages/scipy/sparse/linalg/_dsolve/linsolve.py:653: SparseEfficiencyWarning: splu converted its input to CSC format
  return splu(A).solve
/usr/local/lib/python3.13/dist-packages/scipy/sparse/linalg/_matfuncs.py:706: SparseEfficiencyWarning: spsolve is more efficient when sparse b is in the CSC matrix format
  return spsolve(Q, P)


complex  qubits=2  params=3  exact_2q=-194.32950979  VQE_sv=-194.32950979  |VQE-CASCI|=1.91e-06 kcal/mol  t=1.10s
Pt       qubits=2  params=3  exact_2q=-118.35747572  VQE_sv=-118.35747572  |VQE-CASCI|=7.81e-07 kcal/mol  t=1.23s
H2O      qubits=2  params=3  exact_2q=-75.96110767  VQE_sv=-75.96110767  |VQE-CASCI|=1.86e-07 kcal/mol  t=0.78s
Pt_cp    qubits=2  params=3  exact_2q=-118.35758609  VQE_sv=-118.35758609  |VQE-CASCI|=2.54e-06 kcal/mol  t=0.72s
H2O_cp   qubits=2  params=3  exact_2q=-75.96504248  VQE_sv=-75.96504248  |VQE-CASCI|=7.74e-07 kcal/mol  t=0.53s

Complex Hamiltonian:
 SparsePauliOp(['II', 'IZ', 'IX', 'ZI', 'XI', 'ZZ', 'ZX', 'XZ', 'XX'],
              coeffs=[-0.96212933+0.j,  0.06943204+0.j, -0.01600472+0.j, -0.06943204+0.j,
 -0.01600472+0.j, -0.04282775+0.j,  0.01600484+0.j, -0.01600484+0.j,
  0.02184142+0.j])


In [5]:
def measurement_circuits(circ, qop):
    groups = qop.group_commuting(qubit_wise=True)
    circs = []
    for g in groups:
        qc = circ.copy()
        for q in range(qop.num_qubits):
            letters = {lab[::-1][q] for lab in g.paulis.to_labels()} - {"I"}
            b = letters.pop() if letters else "Z"
            if b == "X":
                qc.h(q)
            elif b == "Y":
                qc.sdg(q); qc.h(q)
        qc.measure_all()
        circs.append(qc)
    return groups, circs

def energy_from_counts(groups, counts_list, e_core):
    E = 0.0
    for g, counts in zip(groups, counts_list):
        nshots = sum(counts.values())
        for lab, coeff in zip(g.paulis.to_labels(), g.coeffs):
            idx = [q for q, c in enumerate(lab[::-1]) if c != "I"]
            ev = sum((-1) ** (sum(int(bits[::-1][q]) for q in idx) % 2) * n
                     for bits, n in counts.items())
            E += coeff.real * ev / nshots
    return E + e_core

for name, d in res.items():
    d["groups"], d["mcircs"] = measurement_circuits(d["circ"], d["qop"])
    print(f"{name:8s} circuits per energy evaluation: {len(d['mcircs'])}")



complex  circuits per energy evaluation: 4
Pt       circuits per energy evaluation: 4
H2O      circuits per energy evaluation: 2
Pt_cp    circuits per energy evaluation: 4
H2O_cp   circuits per energy evaluation: 2


In [7]:
from qbraid.runtime import QbraidProvider
from qiskit import transpile, QuantumCircuit

provider = QbraidProvider(api_key="qbr_815a7d5c0f72cb81dda863ad4a83624f766f38bd2c3a029934996d03d34e3d5e")
for dev in provider.get_devices():
    print(dev.id, dev.status())

DEVICE_ID = "aws:rigetti:qpu:cepheus-1-108q"          # e.g. "rigetti:rigetti:qpu:cepheus-1-108q"
device = provider.get_device(DEVICE_ID)
print(device.metadata())

BASIS_GATES = ["rx", "ry", "rz", "h", "x", "sdg", "cx"]

def run_counts_qbraid(mcircs, theta, shots):
    circs = [transpile(c.assign_parameters(theta), basis_gates=BASIS_GATES, optimization_level=1)
             for c in mcircs]
    t0 = time.time()
    jobs = device.run(circs, shots=shots)
    jobs = jobs if isinstance(jobs, list) else [jobs]
    counts = [job.result().data.get_counts() for job in jobs]
    return counts, time.time() - t0



aws:aqt:qpu:ibex-q1 DeviceStatus.UNAVAILABLE
aws:quera:qpu:aquila DeviceStatus.ONLINE
aws:aws:sim:dm1 DeviceStatus.ONLINE
azure:quantinuum:sim:h2-1e DeviceStatus.OFFLINE
azure:pasqal:qpu:fresnel DeviceStatus.OFFLINE
azure:pasqal:sim:emu-tn DeviceStatus.OFFLINE
aws:iqm:qpu:emerald DeviceStatus.OFFLINE
azure:ionq:qpu:forte-1 DeviceStatus.OFFLINE
aws:aws:sim:sv1 DeviceStatus.ONLINE
aws:ionq:qpu:forte-1 DeviceStatus.ONLINE
ionq:ionq:sim:simulator DeviceStatus.ONLINE
aws:ionq:qpu:forte-enterprise-1 DeviceStatus.OFFLINE
ionq:ionq:qpu:forte-1 DeviceStatus.UNAVAILABLE
qbraid:qbraid:sim:qir-sv DeviceStatus.ONLINE
aws:iqm:qpu:garnet DeviceStatus.UNAVAILABLE
azure:quantinuum:sim:h2-1sc DeviceStatus.OFFLINE
azure:rigetti:sim:qvm DeviceStatus.OFFLINE
azure:rigetti:qpu:cepheus-1-36q DeviceStatus.OFFLINE
azure:ionq:sim:simulator DeviceStatus.OFFLINE
azure:ionq:qpu:forte-enterprise-1 DeviceStatus.OFFLINE
rigetti:rigetti:qpu:cepheus-1-108q DeviceStatus.ONLINE
qbraid:quobly:sim:alloy-forge DeviceStatus.

In [8]:
qc = QuantumCircuit(2)
qc.x(0)
qc.measure_all()
test_counts = device.run(transpile(qc, basis_gates=BASIS_GATES), shots=100).result().data.get_counts()
print("Bit-order test:", test_counts)
REVERSE_BITS = max(test_counts, key=test_counts.get) == "10"
print("Reverse bit order:", REVERSE_BITS)

def fix_order(counts):
    return {k[::-1]: v for k, v in counts.items()} if REVERSE_BITS else counts



Bit-order test: {'00': 10, '01': 88, '10': 1, '11': 1}
Reverse bit order: False


In [9]:
SHOTS = 1000
REPS = 1

for name, d in res.items():
    energies, walls = [], []
    for r in range(REPS):
        counts, wall = run_counts_qbraid(d["mcircs"], d["theta_opt"], SHOTS)
        energies.append(energy_from_counts(d["groups"], [fix_order(c) for c in counts], d["e_core"]))
        walls.append(wall)
    d["hw_all"] = energies
    d["vqe_hw"] = np.mean(energies)
    d["vqe_hw_std"] = np.std(energies, ddof=1)
    d["t_hw_wall"] = np.mean(walls)
    print(f"{name:8s} CASCI={d['e_casci']:.6f}  HW={d['vqe_hw']:.6f} ± {d['vqe_hw_std']:.6f}  "
          f"wall time per evaluation={d['t_hw_wall']:.1f}s")



/usr/local/lib/python3.13/dist-packages/numpy/_core/_methods.py:219: RuntimeWarning: Degrees of freedom <= 0 for slice
  ret = _var(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/usr/local/lib/python3.13/dist-packages/numpy/_core/_methods.py:211: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


complex  CASCI=-194.329510  HW=-194.318418 ± nan  wall time per evaluation=57.5s


/usr/local/lib/python3.13/dist-packages/numpy/_core/_methods.py:219: RuntimeWarning: Degrees of freedom <= 0 for slice
  ret = _var(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/usr/local/lib/python3.13/dist-packages/numpy/_core/_methods.py:211: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


Pt       CASCI=-118.357476  HW=-118.347299 ± nan  wall time per evaluation=55.5s


/usr/local/lib/python3.13/dist-packages/numpy/_core/_methods.py:219: RuntimeWarning: Degrees of freedom <= 0 for slice
  ret = _var(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/usr/local/lib/python3.13/dist-packages/numpy/_core/_methods.py:211: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


H2O      CASCI=-75.961108  HW=-75.905520 ± nan  wall time per evaluation=47.9s


/usr/local/lib/python3.13/dist-packages/numpy/_core/_methods.py:219: RuntimeWarning: Degrees of freedom <= 0 for slice
  ret = _var(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/usr/local/lib/python3.13/dist-packages/numpy/_core/_methods.py:211: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


Pt_cp    CASCI=-118.357586  HW=-118.346072 ± nan  wall time per evaluation=25.0s
H2O_cp   CASCI=-75.965042  HW=-75.886601 ± nan  wall time per evaluation=44.1s


/usr/local/lib/python3.13/dist-packages/numpy/_core/_methods.py:219: RuntimeWarning: Degrees of freedom <= 0 for slice
  ret = _var(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/usr/local/lib/python3.13/dist-packages/numpy/_core/_methods.py:211: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


In [10]:
def e_bind(key, cp=True):
    a, b = ("Pt_cp", "H2O_cp") if cp else ("Pt", "H2O")
    return (res["complex"][key] - res[a][key] - res[b][key]) * H2KCAL

def err_bind(cp=True):
    keys = ["complex", "Pt_cp", "H2O_cp"] if cp else ["complex", "Pt", "H2O"]
    return np.sqrt(sum(res[k]["vqe_hw_std"] ** 2 / REPS for k in keys)) * H2KCAL

def t_sum(key, cp=True):
    keys = ["complex", "Pt_cp", "H2O_cp"] if cp else ["complex", "Pt", "H2O"]
    return sum(res[k][key] for k in keys)

rows = [("HF", "e_hf", "t_hf"), ("HF+VQE 2q (simulator)", "vqe_sv", "t_vqe_sv"),
        ("HF+VQE 2q (hardware, at theta*)", "vqe_hw", "t_hw_wall"),
        ("CASCI(2,2) exact (control)", "e_casci", None), ("CCSD", "e_cc", "t_cc")]

print("=" * 86)
print(f"{'Method':34s}{'E_bind raw':>14s}{'E_bind CP':>18s}{'time CP set (s)':>18s}")
print(f"{'':34s}{'(kcal/mol)':>14s}{'(kcal/mol)':>18s}")
print("-" * 86)
for label, key, tkey in rows:
    raw, cp = e_bind(key, cp=False), e_bind(key, cp=True)
    if key == "vqe_hw":
        raw_s = f"{raw:.2f}±{err_bind(False):.2f}"; cp_s = f"{cp:.2f}±{err_bind(True):.2f}"
    else:
        raw_s, cp_s = f"{raw:.2f}", f"{cp:.2f}"
    t = "" if tkey is None else f"{t_sum(tkey) + (t_sum('t_hf') if key.startswith('vqe') else 0):.2f}"
    print(f"{label:34s}{raw_s:>14s}{cp_s:>18s}{t:>18s}")
print("=" * 86)
print(f"Device: {DEVICE_ID} | shots={SHOTS} | reps={REPS} | basis={BASIS} | spin={SPIN}")


Method                                E_bind raw         E_bind CP   time CP set (s)
                                      (kcal/mol)        (kcal/mol)
--------------------------------------------------------------------------------------
HF                                         -6.00             -3.40              2.52
HF+VQE 2q (simulator)                      -6.86             -4.32              4.86
HF+VQE 2q (hardware, at theta*)       -41.16±nan        -53.81±nan            129.04
CASCI(2,2) exact (control)                 -6.86             -4.32                  
CCSD                                      -18.40            -12.86             32.45
Device: aws:rigetti:qpu:cepheus-1-108q | shots=1000 | reps=1 | basis=def2-SVP | spin=0
